# 二吸収帯整合型推定のための合成メタンプルーム注入・評価

HISUIの実観測背景スペクトルに既知のメタン濃度増分を注入し、1.6 µm帯と2.3 µm帯を用いた推定性能を評価する。

MODTRAN LUTの濃度軸は、**背景からの増分ではなく、大気中メタンの絶対濃度**とする。

背景大気中のメタン濃度を $c_{\mathrm{bg}}$、注入する濃度増分を $\Delta c$ とすると、注入後の絶対濃度は

$$
c_{\mathrm{total}}(x,y)=c_{\mathrm{bg}}+\Delta c(x,y)
$$

です。

合成注入には次の放射輝度比を使う。

$$
T(\lambda,\Delta c)=
\frac{L_{\mathrm{MODTRAN}}(\lambda,c_{\mathrm{bg}}+\Delta c)}
{L_{\mathrm{MODTRAN}}(\lambda,c_{\mathrm{bg}})}
$$

したがって、合成スペクトルは

$$
L_{\mathrm{syn}}(x,y,\lambda)=
L_{\mathrm{bg}}(x,y,\lambda)
T\left(\lambda,\Delta c_{\mathrm{true}}(x,y)\right)
$$

となる。

この比ではMODTRAN放射輝度の一定倍率が分子・分母で相殺されるため、MODTRANを100倍してHISUIと単位を合わせる処理は不要。

## このNotebookの位置づけ

- 注入前HISUI cubeを真の背景として使うOracle評価
- 1.6 µm帯単独の線形MF
- 2.3 µm帯単独の線形MF
- 両帯域に共通の濃度増分を課すLUT探索
- 非線形最小二乗による連続値精密化
- 真値とのRMSE、Bias、$R^2$、検出性能比較

完全なSC-LMMFに必要なスペクトルゲイン、傾き、波長シフトなどの同時推定は、まだ含めていない。

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import least_squares

np.set_printoptions(precision=5, suppress=True)


## 1. 設定

In [ ]:
ROI_CSV = r"E:\refit\all_roi_spectra.csv"
CH4_LUT_CSV = r"E:\refit\CH4b.csv"

# MODTRAN LUTの濃度軸は絶対濃度 [ppm]
BACKGROUND_CH4_PPM = 1.8
MAX_ENHANCEMENT_PPM = 0.7
ENHANCEMENT_STEP_PPM = 0.02

FWHM_NM = 12.5
WINDOW_16 = (1580.0, 1750.0)
WINDOW_23 = (2100.0, 2450.0)
UAS_MAX_ENHANCEMENT_PPM = 0.5

PLUME_PEAK_ENHANCEMENT_PPM = 0.6
PLUME_CENTER_YX = None
PLUME_ANGLE_DEG = 20.0
PLUME_DECAY_PIX = 18.0
PLUME_CROSS_SIGMA_PIX = 4.0
PLUME_SOURCE_SIGMA_PIX = 2.0

## 2. HISUI ROI スペクトルCSVの読み込み

In [ ]:
def get_wave_columns(df):
    pattern = re.compile(r"^wave_([0-9.]+)nm$")
    pairs = []
    for col in df.columns:
        match = pattern.match(str(col))
        if match is not None:
            pairs.append((col, float(match.group(1))))
    if not pairs:
        raise ValueError(f"wave_***nm形式の列が見つかりません。先頭列: {list(df.columns[:10])}")
    pairs.sort(key=lambda item: item[1])
    return [p[0] for p in pairs], np.array([p[1] for p in pairs], dtype=float)


def load_roi_spectra_csv(path):
    df = pd.read_csv(path)
    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("CSVには y と x 列が必要です。")
    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(df, spectra, fill_value=np.nan):
    frame = df.reset_index(drop=True)
    ys = np.sort(frame["y"].unique())
    xs = np.sort(frame["x"].unique())
    y_to_i = {v:i for i,v in enumerate(ys)}
    x_to_i = {v:i for i,v in enumerate(xs)}
    cube = np.full((len(ys), len(xs), spectra.shape[1]), fill_value, dtype=float)
    for row_i, row in frame.iterrows():
        cube[y_to_i[row["y"]], x_to_i[row["x"]]] = spectra[row_i]
    return cube, ys, xs


def make_valid_pixel_mask(cube, nodata_values=(0.0, -9999.0), require_positive=True):
    valid = np.isfinite(cube)
    for value in nodata_values:
        valid &= cube != value
    if require_positive:
        valid &= cube > 0
    return np.all(valid, axis=2)

In [ ]:
df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube_background_true, y_values, x_values = spectra_to_cube(df, spectra)
valid_mask = make_valid_pixel_mask(cube_background_true)

print("DataFrame shape:", df.shape)
print("Cube shape:", cube_background_true.shape)
print("Wavelength range:", wavelengths[0], "to", wavelengths[-1], "nm")
print("Valid pixels:", int(valid_mask.sum()), "/", valid_mask.size)

## 3. MODTRAN CH₄ LUTの読み込みと装置関数畳み込み

In [ ]:
def load_ch4_absolute_concentration_lut(path):
    df_lut = pd.read_csv(path)
    candidates = [c for c in df_lut.columns if str(c).strip().lower() in {"wavelength", "wave", "wavelength_nm"}]
    if not candidates:
        raise ValueError("LUTに wavelength 列が見つかりません。")
    wave_col = candidates[0]
    mod_wave = df_lut[wave_col].to_numpy(dtype=float)

    pairs = []
    for col in df_lut.columns:
        if col == wave_col:
            continue
        try:
            pairs.append((col, float(str(col).strip())))
        except ValueError:
            pass
    if len(pairs) < 2:
        raise ValueError("LUTに絶対メタン濃度を表す数値列が2列以上必要です。")
    pairs.sort(key=lambda item: item[1])
    abs_grid = np.array([p[1] for p in pairs], dtype=float)
    spectra = df_lut[[p[0] for p in pairs]].to_numpy(dtype=float).T
    order = np.argsort(mod_wave)
    return mod_wave[order], abs_grid, spectra[:, order]


def gaussian_srf_resample(mod_wave, mod_spectra, sensor_wave, fwhm_nm):
    mod_wave = np.asarray(mod_wave, float)
    mod_spectra = np.asarray(mod_spectra, float)
    sensor_wave = np.asarray(sensor_wave, float)
    fwhm = np.full(sensor_wave.shape, float(fwhm_nm)) if np.isscalar(fwhm_nm) else np.asarray(fwhm_nm, float)
    if fwhm.shape != sensor_wave.shape:
        raise ValueError("FWHM配列の長さがsensor_waveと一致しません。")

    out = np.full((mod_spectra.shape[0], sensor_wave.size), np.nan)
    for j, center in enumerate(sensor_wave):
        sigma = fwhm[j] / (2.0*np.sqrt(2.0*np.log(2.0)))
        use = np.abs(mod_wave-center) <= 4.0*sigma
        if use.sum() < 2:
            out[:, j] = np.array([np.interp(center, mod_wave, s) for s in mod_spectra])
        else:
            w = np.exp(-0.5*((mod_wave[use]-center)/sigma)**2)
            w /= w.sum()
            out[:, j] = mod_spectra[:, use] @ w
    return out

In [ ]:
modtran_wavelengths, absolute_concentration_grid, modtran_spectra = load_ch4_absolute_concentration_lut(CH4_LUT_CSV)
sensor_lut_absolute = gaussian_srf_resample(
    modtran_wavelengths, modtran_spectra, wavelengths, FWHM_NM
)

print("Absolute CH4 concentration grid [ppm]:", absolute_concentration_grid)
print("Sensor-resolution LUT shape:", sensor_lut_absolute.shape)

## 4. 背景濃度基準の濃度増分LUTを作成

In [ ]:
def interpolate_absolute_lut_spectrum(concentration_ppm, concentration_grid, sensor_lut):
    if not concentration_grid.min() <= concentration_ppm <= concentration_grid.max():
        raise ValueError(
            f"{concentration_ppm:.4f} ppmはLUT範囲外です。LUT範囲: "
            f"{concentration_grid.min():.4f}–{concentration_grid.max():.4f} ppm"
        )
    return np.array([
        np.interp(concentration_ppm, concentration_grid, sensor_lut[:, j])
        for j in range(sensor_lut.shape[1])
    ])


def build_enhancement_grid(background_ppm, max_enhancement_ppm, step_ppm, concentration_grid):
    max_allowed = concentration_grid.max() - background_ppm
    if background_ppm < concentration_grid.min():
        raise ValueError("背景メタン濃度がLUT下限未満です。")
    if max_enhancement_ppm > max_allowed + 1e-12:
        raise ValueError(
            f"背景濃度+最大増分がLUT上限を超えます。使用可能最大増分: {max_allowed:.4f} ppm"
        )
    grid = np.arange(0.0, max_enhancement_ppm + 0.5*step_ppm, step_ppm)
    grid[-1] = min(grid[-1], max_enhancement_ppm)
    return np.unique(np.append(grid, max_enhancement_ppm))


def make_enhancement_ratio_lut(sensor_lut, concentration_grid, background_ppm, enhancement_grid):
    bg = interpolate_absolute_lut_spectrum(background_ppm, concentration_grid, sensor_lut)
    bg = np.maximum(bg, 1e-30)
    ratio_lut = []
    for dc in enhancement_grid:
        enhanced = interpolate_absolute_lut_spectrum(background_ppm + dc, concentration_grid, sensor_lut)
        ratio_lut.append(enhanced / bg)
    return np.asarray(ratio_lut)

In [ ]:
enhancement_grid = build_enhancement_grid(
    BACKGROUND_CH4_PPM,
    MAX_ENHANCEMENT_PPM,
    ENHANCEMENT_STEP_PPM,
    absolute_concentration_grid
)

enhancement_ratio_lut = make_enhancement_ratio_lut(
    sensor_lut_absolute,
    absolute_concentration_grid,
    BACKGROUND_CH4_PPM,
    enhancement_grid
)

print("Background CH4 concentration:", BACKGROUND_CH4_PPM, "ppm")
print("Enhancement grid [ppm]:", enhancement_grid)
print("Enhancement-ratio LUT shape:", enhancement_ratio_lut.shape)

## 5. 合成メタンプルーム濃度増分マップ

In [ ]:
def make_advected_enhancement_plume(shape, peak_enhancement_ppm, center_yx=None,
                                     angle_deg=0.0, decay_length=18.0,
                                     cross_sigma=4.0, source_sigma=2.0):
    h, w = shape
    if center_yx is None:
        center_yx = (h//2, w//2)
    cy, cx = center_yx
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    theta = np.deg2rad(angle_deg)
    dx, dy = xx-cx, yy-cy
    along = dx*np.cos(theta) + dy*np.sin(theta)
    cross = -dx*np.sin(theta) + dy*np.cos(theta)
    downstream = np.maximum(along, 0.0)
    plume = np.exp(-downstream/max(decay_length,1e-6))*np.exp(-0.5*(cross/max(cross_sigma,1e-6))**2)
    plume *= along >= 0
    source = np.exp(-0.5*((dx/max(source_sigma,1e-6))**2 + (dy/max(source_sigma,1e-6))**2))
    plume = np.maximum(plume, source)
    if plume.max() > 0:
        plume /= plume.max()
    return peak_enhancement_ppm * plume


if PLUME_PEAK_ENHANCEMENT_PPM > enhancement_grid.max():
    raise ValueError("合成プルームのピーク増分が増分LUT上限を超えています。")

alpha_true_enhancement = make_advected_enhancement_plume(
    cube_background_true.shape[:2],
    PLUME_PEAK_ENHANCEMENT_PPM,
    PLUME_CENTER_YX,
    PLUME_ANGLE_DEG,
    PLUME_DECAY_PIX,
    PLUME_CROSS_SIGMA_PIX,
    PLUME_SOURCE_SIGMA_PIX
)
alpha_true_enhancement[~valid_mask] = np.nan

plt.figure(figsize=(6,5))
plt.imshow(alpha_true_enhancement)
plt.colorbar(label="Injected CH4 enhancement [ppm]")
plt.title("Synthetic CH4 enhancement plume")
plt.xlabel("x"); plt.ylabel("y"); plt.show()